# MTP Well — Closed-System Aerobic Growth

This notebook models a single microtitre plate (MTP) well as a 0-D gas-liquid
control volume with **no external gas boundaries** — the headspace is sealed and
gas exchange occurs only between the liquid and the fixed headspace volume.

Topology is deliberately minimal:

| Layer | Approach |
|---|---|
| Species | Locally declared (substrate, biomass); inorganics from `common_species` |
| Kinetics | `ReactionBuilder.monod_aerobic_growth` — same model as `Example2_batch_fermenter` |
| Equilibrium chemistry | **Loaded from `AQUEOUS_DEFAULT` database** — no manual declaration |
| Gas-liquid transfer | `EquilibriumTransferModel` — instantaneous (thin film, small volume) |
| Boundaries | None — closed headspace |
| pH control | None — uncontrolled |

Because the headspace is sealed, dissolved oxygen is the limiting resource:
as yeast consume O₂ the headspace fraction drops, growth slows, and the
vapour-phase composition shifts towards CO₂. The bottom plot tracks this
depletion alongside the biomass build-up and pH drift.

## 1 · Environment setup

In [ ]:
import sys
from pathlib import Path

def _find_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError("Run from inside the PyOMES repo")

_root = _find_root()
if str(_root / "models") not in sys.path:
    sys.path.insert(0, str(_root / "models"))

import numpy as np
import matplotlib.pyplot as plt

from PyOMES.chemistry import Species
from PyOMES.chemistry.databases.aqueous import AQUEOUS_DEFAULT
from PyOMES.chemistry.databases.anaerobic_digestion import AD_BASIC
from PyOMES.reactions import (
    EquilibriumReaction, KineticReaction,
    ReactionBuilder, ReactionSystem,
)
from PyOMES.core import (
    ControlVolume, EquilibriumTransferModel, GasPhase,
    LiquidPhase, Simulation,
)
from PyOMES.core.phases import R_L_ATM_MOL_K

print("Imports OK - repo root:", _root)

## 2 · Physical parameters

Deep-well plate geometry: 200 µL liquid in a 700 µL total well volume gives a
500 µL headspace. The larger headspace-to-liquid ratio (relative to a standard
96-well plate) ensures the O₂ depletion dynamics are visible over a 5 h run.

In [ ]:
T_K            = 305.15    # 32 °C — typical yeast fermentation
V_LIQUID_UL    = 200.0     # µL liquid
V_HEADSPACE_UL = 500.0     # µL headspace (sealed)

V_LIQ = V_LIQUID_UL    / 1e6   # L
V_GAS = V_HEADSPACE_UL / 1e6   # L

TAU_H   = 5.0    # simulation duration (h)
N_STEPS = 1000

print(f"V_liq = {V_LIQ*1e6:.0f} µL   V_gas = {V_GAS*1e6:.0f} µL")
print(f"Headspace fraction: {V_GAS/(V_GAS+V_LIQ):.0%}")

## 3 · Species declarations

Only domain-specific species are declared here. Standard inorganics (H⁺, OH⁻,
H₂O, CO₂, HCO₃⁻) are resolved automatically by the string stoichiometry parser
and by `AQUEOUS_DEFAULT`. `YEAST` carries an explicit `MW` because the empirical
CH₁.₆₁O₀.₅₆ formula omits nitrogen.

In [ ]:
ACETIC_ACID   = Species(id="AceticAcid",  atoms={"C":2,"H":4,"O":2},       charge=0)
ACETATE_MINUS = Species(id="Acetate-",    atoms={"C":2,"H":3,"O":2},       charge=-1)
YEAST         = Species(id="Yeast",       atoms={"C":1,"H":1.61,"O":0.56}, charge=0, MW=24.626)

print("Domain species:")
for sp in [ACETIC_ACID, ACETATE_MINUS, YEAST]:
    print(f"  {sp.id:>14}  MW={float(sp.MW):.3f}  charge={sp.charge}")

## 4 · Kinetic reaction: aerobic growth on acetic acid

Identical to `Example2_batch_fermenter` — Monod kinetics with the same
parameters. `ReactionBuilder.monod_aerobic_growth` builds the rate closure
and derives the O₂ / CO₂ / H₂O stoichiometry from elemental balance.

In [ ]:
MU_MAX = 0.5    # 1/h
KS_G_L = 5e-3  # g/L
YIELD  = 0.36   # g biomass / g substrate

rxn_growth = ReactionBuilder.monod_aerobic_growth(
    substrate    = ACETIC_ACID,
    biomass      = YEAST,
    mu_max_per_h = MU_MAX,
    Ks_gL        = KS_G_L,
    yield_gX_gS  = YIELD,
    label        = "growth_on_AceticAcid",
)

print("Kinetic reaction:", rxn_growth.label)
print("Stoichiometry:")
for e in rxn_growth.stoichiometry:
    print(f"  {e.coefficient:+.4g}  {e.species.id}  ({e.phase})")

## 5 · Equilibrium reactions from the package database

`AQUEOUS_DEFAULT` is the standard aqueous-chemistry database shipped with the
package. It bundles three pre-validated equilibrium reactions:

| Label | Reaction | Source |
|---|---|---|
| `eq_water` | H₂O ⇌ H⁺ + OH⁻ | BSM2 / Rosen & Jeppsson 2006 |
| `eq_CO2` | CO₂ + H₂O ⇌ HCO₃⁻ + H⁺ | BSM2 pKa1 = 6.35 |
| `eq_NH4` | NH₄⁺ ⇌ NH₃ + H⁺ | BSM2 pKa = 9.25 |

This model does not include dissolved nitrogen species, so only `eq_water` and
`eq_CO2` are relevant. They are extracted by label — no manual `log_K`,
`dH_J_per_mol`, or `balance_elements` needed.

In [ ]:
print("All reactions in AQUEOUS_DEFAULT:")
for rxn in AQUEOUS_DEFAULT.reactions:
    print(f"  {rxn.label:20s}  log_K={rxn.log_K}  "
          f"dH={rxn.dH_J_per_mol}  species={rxn.species_ids}")

# Extract only the reactions relevant to this model
RELEVANT = {"eq_water", "eq_CO2"}
db_rxns = [r for r in AQUEOUS_DEFAULT.reactions if r.label in RELEVANT]

print(f"\nUsing {len(db_rxns)} database reactions: {[r.label for r in db_rxns]}")

## 6 · Acetic acid dissociation

This reaction is domain-specific (not in any shipped database) so it must be
declared locally. `AceticAcid` and `Acetate-` are passed via `species={...}`
because they are not in `common_species`.

In [ ]:
rxn_dissoc = EquilibriumReaction(
    "AceticAcid,aq <-> Acetate-,aq + H+,aq",
    species={"AceticAcid": ACETIC_ACID, "Acetate-": ACETATE_MINUS},
    log_K=-4.756,
    label="eq_AceticAcid",
)
print(rxn_dissoc.label, f"  log_K = {rxn_dissoc.log_K}")

## 7 · Assemble the ReactionSystem

One kinetic reaction + two database equilibria + one domain equilibrium.
`ReactionSystem` pre-buckets them by type at construction.

In [ ]:
rxn_system = ReactionSystem(
    [rxn_growth] + db_rxns + [rxn_dissoc],
    label="mtp_well_chemistry",
)

print(f"Kinetic reactions      : {len(rxn_system.kinetic_reactions)}")
print(f"Single-phase equilibria: {len(rxn_system.single_phase_equilibria)}")
print(f"  {[r.label for r in rxn_system.single_phase_equilibria]}")
print(f"Species                : {', '.join(rxn_system.species_ids)}")

## 8 · Build the gas and liquid phases

The gas phase is initialised with dry air (21 % O₂, 79 % N₂, trace CO₂).
The initial dissolved-gas concentrations are set to Henry equilibrium with the
headspace. Substrate and inoculum are set from target mass concentrations.

In [ ]:
# Gas phase — dry air at 1 atm
n_total_gas = (1.0 * V_GAS) / (R_L_ATM_MOL_K * T_K)  # PV = nRT

gas = GasPhase(
    n_mol={
        "O2":  n_total_gas * 0.2095,
        "CO2": n_total_gas * 0.0004,
        "N2":  n_total_gas * 0.7901,
    },
    V_L=V_GAS,
    T_K=T_K,
)

print(f"Total gas moles: {n_total_gas:.4e} mol")
for sp, n in gas.n_mol.items():
    print(f"  {sp:>4}: {n:.4e} mol")

# Dissolved gases at Henry equilibrium
def _henry_n(sp):
    pm = AD_BASIC.partition_models[sp]
    return pm.H_ref * gas.p_atm.get(sp, 0.0) * 101325 / 1000 * V_LIQ

# Initial substrate / biomass
C_ACETATE_0_G_L = 0.5    # g/L total acetic acid
C_YEAST_0_G_L   = 0.02   # g/L inoculum
n_acetate_0 = (C_ACETATE_0_G_L / float(ACETIC_ACID.MW)) * V_LIQ
n_yeast_0   = (C_YEAST_0_G_L   / float(YEAST.MW))        * V_LIQ

liquid = LiquidPhase(
    n_mol={
        ACETIC_ACID.id:   n_acetate_0,
        ACETATE_MINUS.id: 0.0,
        YEAST.id:         n_yeast_0,
        "O2":             _henry_n("O2"),
        "CO2":            _henry_n("CO2"),
        "N2":             _henry_n("N2"),
        "HCO3-":          0.0,
        "OH-":            0.0,
        "H+":             1.0e-7 * V_LIQ,
    },
    V_L=V_LIQ,
    T_K=T_K,
)

print(f"\nInitial liquid ({V_LIQ*1e6:.0f} µL):")
print(f"  AceticAcid: {C_ACETATE_0_G_L:.2f} g/L")
print(f"  Yeast:      {C_YEAST_0_G_L:.3f} g/L")

## 9 · Transfer models

In a microtitre well the gas-liquid film is thin (< 0.5 mm) and well-mixed at
the microlitre scale. `EquilibriumTransferModel` treats the gas-liquid
exchange as instantaneous — no kLa parameter is needed. `AD_BASIC` supplies
the Henry constants for O₂, CO₂, and N₂.

Because there is no gas feed or vent, the **total moles of each species in
gas + liquid are conserved** — this is the closed-headspace constraint.

In [ ]:
transfer_models = {
    "O2":  EquilibriumTransferModel(AD_BASIC.partition_models["O2"]),
    "CO2": EquilibriumTransferModel(AD_BASIC.partition_models["CO2"]),
    "N2":  EquilibriumTransferModel(AD_BASIC.partition_models["N2"]),
}

print("Transfer models (all equilibrium — closed headspace):")
for sp, tm in transfer_models.items():
    pm = tm.partition_model
    print(f"  {sp:>4}  H_ref={pm.H_ref:.2e} mol/(m3·Pa)  dlnH={pm.dlnH:.0f} K")

## 10 · Assemble the ControlVolume

No `boundaries` are added — the well is sealed. The CV manages only the
internal gas-liquid exchange and the reaction / speciation steps.

In [ ]:
cv = ControlVolume(
    phases          = {"gas": gas, "liquid": liquid},
    transfer_models = transfer_models,
    reaction_system = rxn_system,
    label           = "mtp_well",
)

print("CV label  :", cv.label)
print("Phases    :", list(cv.phases.keys()))
print("Boundaries:", len(cv.boundaries), "(none — closed system)")

## 11 · Build the Simulation

In [ ]:
sim = Simulation(
    cvs   = {"main": cv},
    label = "mtp_well_sim",
)
print("Simulation:", sim.label)

## 12 · Run

In [ ]:
result = sim.run(tau_h=TAU_H, n_steps=N_STEPS)
print(f"Finished in {result.runtime_s:.2f} s")

## 13 · Summary

In [ ]:
liq = result.liquid_mol["main"]
gas_mol = result.gas_mol["main"]

MW_S = float(ACETIC_ACID.MW)
MW_X = float(YEAST.MW)

C_S0 = liq["AceticAcid"][0]  / V_LIQ * MW_S
C_Sf = liq["AceticAcid"][-1] / V_LIQ * MW_S
C_X0 = liq["Yeast"][0]       / V_LIQ * MW_X
C_Xf = liq["Yeast"][-1]      / V_LIQ * MW_X

pH = result.pH["main"]
ph_valid = pH[np.isfinite(pH)]

print(f"Substrate:  {C_S0:.3f} -> {C_Sf:.3f} g/L  (consumed: {C_S0-C_Sf:.3f} g/L)")
print(f"Biomass:    {C_X0:.4f} -> {C_Xf:.4f} g/L")
print(f"pH:         {ph_valid[0]:.3f} -> {ph_valid[-1]:.3f}")

n_O2_0 = gas_mol["O2"][0]  + liq["O2"][0]
n_O2_f = gas_mol["O2"][-1] + liq["O2"][-1]
print(f"Total O2:   {n_O2_0:.4e} -> {n_O2_f:.4e} mol  (fraction remaining: {n_O2_f/n_O2_0:.1%})")

## 14 · Time-series plots

Two-panel single-column figure:
- **Top**: vapour-phase mole fractions of O₂, CO₂, and N₂ over time — shows
  O₂ depletion and CO₂ accumulation in the sealed headspace.
- **Bottom**: substrate and biomass concentrations (left axis, g/L) and pH
  (right axis) over time.

In [ ]:
t       = result.t_h
liq     = result.liquid_mol["main"]
gas_mol = result.gas_mol["main"]
pH      = result.pH["main"]

MW_S = float(ACETIC_ACID.MW)
MW_X = float(YEAST.MW)

# Vapour mole fractions
n_gas_total = gas_mol["O2"] + gas_mol["CO2"] + gas_mol["N2"]
y_O2  = gas_mol["O2"]  / n_gas_total
y_CO2 = gas_mol["CO2"] / n_gas_total
y_N2  = gas_mol["N2"]  / n_gas_total

# Concentrations (g/L)
C_S = liq["AceticAcid"] / V_LIQ * MW_S
C_X = liq["Yeast"]      / V_LIQ * MW_X

# ── Figure ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(8, 9), sharex=True)
fig.suptitle("MTP Well — Closed-System Aerobic Growth on Acetic Acid",
             fontsize=13, y=0.98)

# Top panel: vapour fractions
ax1 = axes[0]
ax1.plot(t, y_O2  * 100, color="tab:blue",   label="O₂")
ax1.plot(t, y_CO2 * 100, color="tab:red",    label="CO₂")
ax1.plot(t, y_N2  * 100, color="tab:gray",   label="N₂", linestyle="--")
ax1.set_ylabel("Vapour mole fraction (%)")
ax1.set_title("Headspace composition")
ax1.legend(loc="center right")
ax1.grid(True, alpha=0.3)

# Bottom panel: concentrations (left) + pH (right)
ax2 = axes[1]
ax2_ph = ax2.twinx()

ax2.plot(t, C_S, color="tab:orange", label="Acetic acid (g/L)")
ax2.plot(t, C_X, color="tab:green",  label="Yeast (g/L)")
ax2_ph.plot(t, pH, color="tab:purple", linestyle=":", label="pH", linewidth=1.5)

ax2.set_xlabel("Time (h)")
ax2.set_ylabel("Concentration (g/L)")
ax2_ph.set_ylabel("pH")
ax2.set_title("Substrate, biomass, and pH")

# Combined legend
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_ph.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, loc="center right")
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()